# Batch Normalization Basics

Batch normalization is a technique used to improve the performance and stability of neural networks. This notebook explores the fundamentals of batch normalization, its implementation, and practical applications in deep learning models.

## Import Required Libraries

Let's import the essential libraries needed for implementing and visualizing batch normalization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import seaborn as sns

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set plotting style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

## Understanding Batch Normalization

### What is Batch Normalization?

Batch normalization is a technique introduced by Sergey Ioffe and Christian Szegedy in their 2015 paper, "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift." 

### The Problem: Internal Covariate Shift

During the training of deep neural networks, the distribution of each layer's inputs changes as the parameters of the previous layers change. This phenomenon is called **internal covariate shift**. This can slow down training by requiring lower learning rates and careful parameter initialization.

### The Solution: Batch Normalization

Batch normalization addresses this issue by normalizing the inputs of each layer for each mini-batch. It does this by:
1. Calculating the mean and variance of the mini-batch
2. Normalizing the inputs using these statistics
3. Scaling and shifting the results using learnable parameters

This technique has several benefits:
- It allows for higher learning rates
- Reduces the strong dependence on initialization
- Acts as a form of regularization
- Improves gradient flow through the network

### Mathematical Formulation

For a layer with inputs $x$ over a mini-batch $\mathcal{B}$, batch normalization performs:

1. Calculate mini-batch mean: 
   $\mu_\mathcal{B} = \frac{1}{m}\sum_{i=1}^{m} x_i$

2. Calculate mini-batch variance: 
   $\sigma_\mathcal{B}^2 = \frac{1}{m}\sum_{i=1}^{m} (x_i - \mu_\mathcal{B})^2$

3. Normalize: 
   $\hat{x}_i = \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2 + \epsilon}}$

4. Scale and shift (learnable parameters): 
   $y_i = \gamma \hat{x}_i + \beta$

Where:
- $\gamma$ (gamma) and $\beta$ (beta) are learnable parameters
- $\epsilon$ is a small constant added for numerical stability

In [ ]:
# Let's implement batch normalization from scratch to understand it better

def batch_norm_manual(x, gamma, beta, eps=1e-5):
    # Calculate batch mean and variance
    batch_mean = np.mean(x, axis=0)
    batch_var = np.var(x, axis=0)
    
    # Normalize
    x_norm = (x - batch_mean) / np.sqrt(batch_var + eps)
    
    # Scale and shift
    out = gamma * x_norm + beta
    
    return out

# Generate some random data
np.random.seed(42)
x = np.random.randn(100, 10)  # 100 samples, 10 features
gamma = np.ones(10)  # Initial scale parameter set to 1
beta = np.zeros(10)  # Initial shift parameter set to 0

# Apply batch normalization
normalized_x = batch_norm_manual(x, gamma, beta)

# Visualize the distribution before and after normalization
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(x[:, 0], bins=30, alpha=0.7)
plt.title('Before Batch Normalization')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.hist(normalized_x[:, 0], bins=30, alpha=0.7, color='green')
plt.title('After Batch Normalization')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

# Check mean and variance
print(f"Before normalization - Mean: {np.mean(x[:, 0]):.4f}, Variance: {np.var(x[:, 0]):.4f}")
print(f"After normalization - Mean: {np.mean(normalized_x[:, 0]):.4f}, Variance: {np.var(normalized_x[:, 0]):.4f}")

## Implementation of Batch Normalization

Now that we understand the theory behind batch normalization, let's see how it's implemented in modern deep learning frameworks like TensorFlow/Keras.

Batch normalization layers can be added after dense or convolutional layers, typically before the activation function.

In [ ]:
# Creating a simple neural network with batch normalization using TensorFlow/Keras
model_with_bn = keras.Sequential([
    layers.Dense(64, kernel_initializer='he_normal', use_bias=False),  # No bias needed with BatchNorm
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    layers.Dense(64, kernel_initializer='he_normal', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    layers.Dense(10, activation='softmax')
])

# Print the model summary
model_with_bn.build((None, 784))  # Build model with input shape for MNIST
model_with_bn.summary()

### Inspecting the BatchNormalization Layer

Let's look more closely at the BatchNormalization layer in TensorFlow to understand its parameters:

1. **momentum**: Controls the moving average for the mean and variance used during inference
2. **epsilon**: Small constant added for numerical stability
3. **center**: If True, add offset (beta) after normalization
4. **scale**: If True, multiply by scale (gamma) after normalization
5. **trainable**: Whether gamma and beta should be updated during training
6. **moving_mean_initializer/moving_variance_initializer**: Initializers for the moving statistics

In [ ]:
# Creating a BatchNormalization layer with custom parameters
custom_bn_layer = layers.BatchNormalization(
    momentum=0.9,            # Default is 0.99
    epsilon=1e-5,            # Default is 1e-3
    center=True,             # Use beta
    scale=True,              # Use gamma
    beta_initializer='zeros',
    gamma_initializer='ones',
    moving_mean_initializer='zeros',
    moving_variance_initializer='ones'
)

# Inspect the layer
print("BatchNormalization Layer Configuration:")
for key, value in custom_bn_layer.get_config().items():
    print(f"{key}: {value}")

## Batch Normalization in Practice

Let's explore several use cases and best practices for batch normalization in neural networks:

1. **Batch Normalization with Dense Layers**
2. **Batch Normalization with Convolutional Layers**
3. **Batch Normalization Placement** (before vs. after activation)

In [ ]:
# 1. Batch Normalization with Dense Layers
dense_model = keras.Sequential([
    layers.InputLayer(input_shape=(784,)),
    
    # Option 1: BN after Dense, before Activation
    layers.Dense(128, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    # Option 2: Combined with Activation
    layers.Dense(64, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    layers.Dense(10, activation='softmax')
])

# 2. Batch Normalization with Convolutional Layers
conv_model = keras.Sequential([
    layers.InputLayer(input_shape=(28, 28, 1)),
    
    # Conv with BN
    layers.Conv2D(32, kernel_size=3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(pool_size=2),
    
    layers.Conv2D(64, kernel_size=3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(pool_size=2),
    
    layers.Flatten(),
    layers.Dense(128, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dense(10, activation='softmax')
])

# Print model summaries
print("Dense Model with Batch Normalization:")
dense_model.summary()

print("\nConvolutional Model with Batch Normalization:")
conv_model.summary()

### Batch Normalization Placement: Before or After Activation?

There's been debate about whether batch normalization should be applied before or after the activation function. The original paper placed it before activation, but some research suggests placing it after can also be effective.

Let's implement both approaches to see the difference:

In [ ]:
# BN before activation (original approach)
model_bn_before_act = keras.Sequential([
    layers.InputLayer(input_shape=(784,)),
    layers.Dense(64, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dense(10, activation='softmax')
])

# BN after activation (alternative approach)
model_bn_after_act = keras.Sequential([
    layers.InputLayer(input_shape=(784,)),
    layers.Dense(64, use_bias=True),  # Bias can be useful here
    layers.Activation('relu'),
    layers.BatchNormalization(),
    layers.Dense(10, activation='softmax')
])

# Note: The original BatchNorm paper and most implementations place it before activation
print("Generally recommended approach: BatchNorm before activation")

## Effects of Batch Normalization

Batch normalization provides several benefits to neural network training:

1. **Faster convergence**: Allows training with higher learning rates
2. **Reduced sensitivity to initialization**: Makes networks more robust to poor initialization
3. **Regularization effect**: Acts as a form of regularization, reducing the need for dropout
4. **Stabilized training**: Reduces the problem of vanishing/exploding gradients

Let's demonstrate these effects by comparing networks with and without batch normalization.

In [ ]:
# Let's load the MNIST dataset to test our models
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Preprocess the data
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

# Create models with and without batch normalization
def create_model(use_batch_norm=False):
    model = keras.Sequential()
    model.add(layers.Conv2D(32, kernel_size=3, padding='same', input_shape=(28, 28, 1)))
    
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(pool_size=2))
    
    model.add(layers.Conv2D(64, kernel_size=3, padding='same'))
    
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(pool_size=2))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(128))
    
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    
    model.add(layers.Activation('relu'))
    model.add(layers.Dense(10, activation='softmax'))
    
    return model

# Initialize models
model_with_bn = create_model(use_batch_norm=True)
model_without_bn = create_model(use_batch_norm=False)

# Compile both models - notice we can use a higher learning rate with batch normalization
model_with_bn.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

model_without_bn.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])

# For time constraints, we'll just train on a subset of data
# Define a small subset for demonstration
x_train_small = x_train[:5000]
y_train_small = y_train[:5000]
x_test_small = x_test[:1000]
y_test_small = y_test[:1000]

In [ ]:
# Train both models and store training history
history_with_bn = model_with_bn.fit(
    x_train_small, y_train_small,
    batch_size=128,
    epochs=5,
    validation_split=0.1,
    verbose=1
)

history_without_bn = model_without_bn.fit(
    x_train_small, y_train_small,
    batch_size=128,
    epochs=5,
    validation_split=0.1,
    verbose=1
)

# Plot training curves to compare performance
plt.figure(figsize=(12, 5))

# Plot training & validation accuracy values
plt.subplot(1, 2, 1)
plt.plot(history_with_bn.history['accuracy'])
plt.plot(history_with_bn.history['val_accuracy'])
plt.plot(history_without_bn.history['accuracy'])
plt.plot(history_without_bn.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['With BN - Train', 'With BN - Validation', 
            'Without BN - Train', 'Without BN - Validation'])

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history_with_bn.history['loss'])
plt.plot(history_with_bn.history['val_loss'])
plt.plot(history_without_bn.history['loss'])
plt.plot(history_without_bn.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['With BN - Train', 'With BN - Validation', 
            'Without BN - Train', 'Without BN - Validation'])

plt.tight_layout()
plt.show()

# Evaluate models on test data
test_loss_bn, test_acc_bn = model_with_bn.evaluate(x_test_small, y_test_small, verbose=0)
test_loss, test_acc = model_without_bn.evaluate(x_test_small, y_test_small, verbose=0)

print(f"Test accuracy with batch normalization: {test_acc_bn:.4f}")
print(f"Test accuracy without batch normalization: {test_acc:.4f}")

## Batch Normalization at Training vs. Inference

Batch normalization operates differently during training versus inference:

### During Training:
- Uses the mean and variance of the current mini-batch
- Updates the running mean and variance for use during inference
- Applies normalization, scaling, and shifting to each batch

### During Inference (Testing):
- Uses the running averages of mean and variance collected during training
- This ensures consistent outputs regardless of batch size (even for single examples)
- Still applies the learned scaling (gamma) and shifting (beta) parameters

Let's demonstrate this difference:

In [ ]:
# Let's create a simple model with batch normalization
simple_model = keras.Sequential([
    layers.Dense(10, input_shape=(5,)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dense(1)
])

# Compile the model
simple_model.compile(optimizer='adam', loss='mse')

# Generate synthetic data
np.random.seed(42)
X = np.random.normal(0, 1, size=(1000, 5))
y = np.sum(X, axis=1, keepdims=True)

# Train the model for a few epochs
simple_model.fit(X, y, epochs=3, batch_size=32, verbose=0)

# Now let's access the BatchNormalization layer
bn_layer = simple_model.layers[1]

# Get the moving mean and variance (used during inference)
moving_mean = bn_layer.moving_mean.numpy()
moving_var = bn_layer.moving_variance.numpy()

# Generate a batch for demonstration
batch_X = X[:32]
batch_y = y[:32]

# Get the batch statistics by running in training mode
with tf.GradientTape() as tape:
    # Set training=True to use batch statistics
    outputs = simple_model(batch_X, training=True)
    
# Calculate batch mean and variance manually
batch_mean = np.mean(batch_X @ simple_model.layers[0].weights[0].numpy(), axis=0)
batch_var = np.var(batch_X @ simple_model.layers[0].weights[0].numpy(), axis=0)

print("Batch Normalization Statistics:")
print(f"Moving (inference) mean: {moving_mean[:3]}...")
print(f"Moving (inference) variance: {moving_var[:3]}...")
print(f"Current batch mean: {batch_mean[:3]}...")
print(f"Current batch variance: {batch_var[:3]}...")

# Let's test the difference between training and inference mode
# Same input, different modes
test_input = np.random.normal(0, 1, size=(1, 5))

# Training mode (uses batch statistics - but meaningless for a single example)
training_output = simple_model(test_input, training=True)

# Inference mode (uses moving statistics)
inference_output = simple_model(test_input, training=False)

print("\nOutputs for the same input:")
print(f"Training mode output: {training_output.numpy().flatten()}")
print(f"Inference mode output: {inference_output.numpy().flatten()}")
print(f"Are they the same? {'Yes' if np.allclose(training_output, inference_output) else 'No'}")

## Visualization of Batch Normalization Effects

To better understand how batch normalization affects the network training, let's visualize:
1. The distribution of activations before and after batch normalization
2. How batch normalization stabilizes these distributions over training iterations

In [ ]:
# Create a model with hooks to extract intermediate activations
def create_visualization_model():
    inputs = keras.Input(shape=(784,))
    
    # First layer without BN
    x1 = layers.Dense(100, activation=None)(inputs)
    
    # Store the pre-BN activations
    pre_bn = x1
    
    # Apply BN
    x2 = layers.BatchNormalization()(x1)
    
    # Store the post-BN activations
    post_bn = x2
    
    # Apply activation
    x3 = layers.Activation('relu')(x2)
    
    # Output layer
    outputs = layers.Dense(10, activation='softmax')(x3)
    
    # Create the model
    model = keras.Model(inputs=inputs, outputs=outputs)
    
    # Create extraction models
    pre_bn_model = keras.Model(inputs=inputs, outputs=pre_bn)
    post_bn_model = keras.Model(inputs=inputs, outputs=post_bn)
    
    return model, pre_bn_model, post_bn_model

# Create the models
main_model, pre_bn_model, post_bn_model = create_visualization_model()

# Compile the main model
main_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Prepare MNIST data for visualization
(x_train, y_train), _ = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype('float32') / 255.0

# Select a small subset for visualization
visualization_data = x_train[:1000]

# Function to plot activation distributions
def plot_activation_distributions(pre_activations, post_activations, epoch):
    plt.figure(figsize=(12, 5))
    
    # Plot pre-BN activations
    plt.subplot(1, 2, 1)
    for i in range(min(5, pre_activations.shape[1])):  # Plot first 5 neurons
        sns.kdeplot(pre_activations[:, i], fill=True, alpha=0.3, label=f'Neuron {i+1}')
    plt.title(f'Pre-Batch Normalization Activations (Epoch {epoch})')
    plt.xlabel('Activation Value')
    plt.ylabel('Density')
    plt.legend()
    
    # Plot post-BN activations
    plt.subplot(1, 2, 2)
    for i in range(min(5, post_activations.shape[1])):  # Plot first 5 neurons
        sns.kdeplot(post_activations[:, i], fill=True, alpha=0.3, label=f'Neuron {i+1}')
    plt.title(f'Post-Batch Normalization Activations (Epoch {epoch})')
    plt.xlabel('Activation Value')
    plt.ylabel('Density')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Get initial activations (before training)
pre_bn_activations = pre_bn_model.predict(visualization_data)
post_bn_activations = post_bn_model.predict(visualization_data)

# Plot initial distributions
plot_activation_distributions(pre_bn_activations, post_bn_activations, epoch=0)

# Train for one epoch
main_model.fit(x_train, y_train, batch_size=128, epochs=1, verbose=1)

# Get activations after one epoch of training
pre_bn_activations = pre_bn_model.predict(visualization_data)
post_bn_activations = post_bn_model.predict(visualization_data)

# Plot distributions after training
plot_activation_distributions(pre_bn_activations, post_bn_activations, epoch=1)

# Calculate statistics
pre_bn_mean = np.mean(pre_bn_activations, axis=0)
pre_bn_std = np.std(pre_bn_activations, axis=0)
post_bn_mean = np.mean(post_bn_activations, axis=0)
post_bn_std = np.std(post_bn_activations, axis=0)

print("\nActivation Statistics:")
print(f"Pre-BN Mean (first 5 neurons): {pre_bn_mean[:5]}")
print(f"Pre-BN Std (first 5 neurons): {pre_bn_std[:5]}")
print(f"Post-BN Mean (first 5 neurons): {post_bn_mean[:5]}")
print(f"Post-BN Std (first 5 neurons): {post_bn_std[:5]}")

## Implementing a Simple Neural Network with Batch Normalization

Let's put everything together and build a complete neural network with batch normalization to classify the MNIST dataset.

In [ ]:
# Prepare MNIST data
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Create a ConvNet with batch normalization
model = keras.Sequential([
    # Input layer
    layers.InputLayer(input_shape=(28, 28, 1)),
    
    # First Conv block with BatchNorm
    layers.Conv2D(32, kernel_size=3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(pool_size=2),
    
    # Second Conv block with BatchNorm
    layers.Conv2D(64, kernel_size=3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(pool_size=2),
    
    # Third Conv block with BatchNorm
    layers.Conv2D(128, kernel_size=3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(pool_size=2),
    
    # Flatten and Dense layers
    layers.Flatten(),
    layers.Dense(128, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    # Output layer
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model summary
model.summary()

# Define callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1)
]

# Train the model (with a small subset for time constraints)
x_train_small = x_train[:10000]
y_train_small = y_train[:10000]
x_val = x_train[10000:12000]
y_val = y_train[10000:12000]

history = model.fit(
    x_train_small, y_train_small,
    validation_data=(x_val, y_val),
    batch_size=128,
    epochs=5,
    callbacks=callbacks,
    verbose=1
)

# Evaluate on test data
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

# Plot training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])

plt.tight_layout()
plt.show()

## Comparison: Networks With and Without Batch Normalization

Finally, let's run a direct comparison between two identical networks - one with batch normalization and one without - to clearly demonstrate the benefits.

In [ ]:
def create_comparison_model(use_batch_norm=False, learning_rate=0.001):
    model = keras.Sequential()
    
    # Input layer
    model.add(layers.InputLayer(input_shape=(28, 28, 1)))
    
    # First Conv block
    model.add(layers.Conv2D(32, kernel_size=3, padding='same', use_bias=not use_batch_norm))
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(pool_size=2))
    
    # Second Conv block
    model.add(layers.Conv2D(64, kernel_size=3, padding='same', use_bias=not use_batch_norm))
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(pool_size=2))
    
    # Dense layers
    model.add(layers.Flatten())
    model.add(layers.Dense(128, use_bias=not use_batch_norm))
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.5))
    
    # Output layer
    model.add(layers.Dense(10, activation='softmax'))
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create both models
model_with_bn = create_comparison_model(use_batch_norm=True, learning_rate=0.001)
model_without_bn = create_comparison_model(use_batch_norm=False, learning_rate=0.001)

# For quicker demonstration, use a subset of data
x_train_subset = x_train[:5000]
y_train_subset = y_train[:5000]
x_val_subset = x_train[5000:6000]
y_val_subset = y_train[5000:6000]

# Train both models for the same number of epochs
history_with_bn = model_with_bn.fit(
    x_train_subset, y_train_subset,
    validation_data=(x_val_subset, y_val_subset),
    batch_size=64,
    epochs=5,
    verbose=1
)

history_without_bn = model_without_bn.fit(
    x_train_subset, y_train_subset,
    validation_data=(x_val_subset, y_val_subset),
    batch_size=64,
    epochs=5,
    verbose=1
)

# Evaluate on test data
test_loss_bn, test_acc_bn = model_with_bn.evaluate(x_test, y_test, verbose=0)
test_loss, test_acc = model_without_bn.evaluate(x_test, y_test, verbose=0)

print(f"Test accuracy with batch normalization: {test_acc_bn:.4f}")
print(f"Test accuracy without batch normalization: {test_acc:.4f}")

# Plot training curves to compare performance
plt.figure(figsize=(15, 10))

# Plot training & validation accuracy values
plt.subplot(2, 2, 1)
plt.plot(history_with_bn.history['accuracy'])
plt.plot(history_with_bn.history['val_accuracy'])
plt.title('Model with Batch Normalization: Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='lower right')

plt.subplot(2, 2, 2)
plt.plot(history_without_bn.history['accuracy'])
plt.plot(history_without_bn.history['val_accuracy'])
plt.title('Model without Batch Normalization: Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='lower right')

# Plot training & validation loss values
plt.subplot(2, 2, 3)
plt.plot(history_with_bn.history['loss'])
plt.plot(history_with_bn.history['val_loss'])
plt.title('Model with Batch Normalization: Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper right')

plt.subplot(2, 2, 4)
plt.plot(history_without_bn.history['loss'])
plt.plot(history_without_bn.history['val_loss'])
plt.title('Model without Batch Normalization: Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper right')

plt.tight_layout()
plt.show()

# Plot direct comparison
plt.figure(figsize=(12, 5))

# Plot training accuracy comparison
plt.subplot(1, 2, 1)
plt.plot(history_with_bn.history['accuracy'])
plt.plot(history_without_bn.history['accuracy'])
plt.title('Training Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['With Batch Norm', 'Without Batch Norm'])

# Plot validation loss comparison
plt.subplot(1, 2, 2)
plt.plot(history_with_bn.history['val_loss'])
plt.plot(history_without_bn.history['val_loss'])
plt.title('Validation Loss Comparison')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['With Batch Norm', 'Without Batch Norm'])

plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've explored batch normalization in depth, including:

1. The theory behind batch normalization and how it helps reduce internal covariate shift
2. The mathematical formulation of batch normalization
3. How to implement batch normalization in neural networks
4. The differences in behavior during training and inference
5. Visualizing the effects of batch normalization on layer activations
6. Comparing the performance of networks with and without batch normalization

Key takeaways:
- Batch normalization stabilizes the learning process and allows for higher learning rates
- It reduces dependency on careful parameter initialization
- It has a regularizing effect that can improve generalization
- It helps gradients flow better through deep networks
- It accelerates training by reducing the number of epochs needed to achieve good performance

Batch normalization has become a standard component in most modern neural network architectures, especially in deep networks where training stability is crucial.